In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Load dataset (download from Kaggle link in README)

df = pd.read_csv("IMDB Dataset.csv")

In [ ]:
df = pd.read_csv(path)

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
df.shape

(49582, 2)

# **Pre-Processing**

In [ ]:
# Converting to lower case

df["review"] = df["review"].str.lower()

In [ ]:
# Removing URLs

import re

def remove_urls(text):
  text = re.sub(r"http\S+","",text)
  return text

df["review"] = df["review"].apply(remove_urls)

In [ ]:
# Removing Punctuations

def remove_punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text)
  return text

df["review"] = df["review"].apply(remove_punctuations)


In [ ]:
# Removing html

def remove_html(text):
  text = re.sub(r"<.*?>","",text)
  return text

df["review"] = df["review"].apply(remove_html)

In [ ]:
# Removing the StopWords

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
def remove_stopwords(text):
  token = word_tokenize(text)
  stop_words = stopwords.words("english")        # returns all words that occurs very frequently and that does not add much value (english lang.)

  for word in token:
    if word in stop_words:
      text = text.replace(word,"")

  return text


df["review"] = df["review"].apply(remove_stopwords)


In [ ]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [ ]:
# Stemming

from nltk import PorterStemmer

def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []

  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token  = ps.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)


In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [ ]:
# Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
y


,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [ ]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [ ]:
# Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [ ]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057140 stored elements and shape (49582, 5000)>

**Dataset and DataLoader**

In [ ]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X,y, test_size = 0.2 , random_state = 42
)

In [ ]:
from torch.utils.data import TensorDataset , DataLoader

In [ ]:
# converting sparse matrix into numpy array

X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
type(X_train)

numpy.ndarray

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set,shuffle = True  , batch_size = 64)
test_loader = DataLoader(test_set,shuffle = True  , batch_size = 64)

# **Build RNN**

In [ ]:
class RNN(nn.Module):
  def __init__(self,input_size , hidden_size = 128 , num_layers = 1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

     # RNN layer
    self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first = True)

    # fully connected layer
    self.fc = nn.Linear(hidden_size , 1)


  def forward(self,x):
    #optional step => shape(no. of layers , batch size , hidden size)
    h0 =  torch.zeros(self.num_layers,x.size(0), self.hidden_size)

    out,_ = self.rnn(x,h0)
    out = self.fc(out[:,-1,:])
    return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterian = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# **Training out RNN**

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for Xb,yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)     # add singleton direction

    outputs = model(Xb)    # (batch_size,1)

    outputs = torch.sigmoid(outputs.squeeze(1))   #(batch_size,)     outputs => Probability
    loss = criterian(outputs,yb)
    loss.backward()
    optimizer.step()

  print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3015923500061035
epoch = 2/10 and loss = 0.39597204327583313
epoch = 3/10 and loss = 0.29216253757476807
epoch = 4/10 and loss = 0.26897159218788147
epoch = 5/10 and loss = 0.1336766630411148
epoch = 6/10 and loss = 0.25641876459121704
epoch = 7/10 and loss = 0.15409085154533386
epoch = 8/10 and loss = 0.16598720848560333
epoch = 9/10 and loss = 0.25799641013145447
epoch = 10/10 and loss = 0.422048419713974


In [ ]:
# evaluate

model.eval()

with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for Xb,yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze(1)) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals  * 100}")


accuracy = 85.57023293334677
